<a href="https://colab.research.google.com/github/serelk/nebius-academy-course/blob/main/week3/task-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community

!pip install pypdf
!pip install -qU langchain-community faiss-cpu
!pip install -U langchain-huggingface

In [ ]:
from datasets import load_dataset
from google.colab import userdata

In [ ]:
dataset = load_dataset("PatronusAI/financebench",)

In [ ]:
df = dataset["train"].to_pandas()
filtered_df = df.where(df.question_type == "domain-relevant").dropna().sort_values(by="financebench_id").head(5)
# filtered_df["evidence"].iloc[0][0]["evidence_page_num"]
filtered_df

In [ ]:
file_path = "https://d18rn0p25nwr6d.cloudfront.net/CIK-0000024741/eb24002c-e050-45eb-806a-15264d59131c.pdf"
loader = PyPDFLoader(file_path)

In [ ]:

docs = loader.load()
page_text = docs[0].page_content
b

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
embedding_dim = 384  # Dimension for 'BAAI/bge-small-en-v1.5'
index = faiss.IndexFlatL2(embedding_dim)

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS


vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set the chunk size to very small. These settings are for illustrative purposes only.
    chunk_size=250,
    # Sets the number of overlapping characters between chunks.
    chunk_overlap=50,
    # Specifies a function to calculate the length of the string.
    length_function=len,
    # Sets whether to use regular expressions as delimiters.
    is_separator_regex=False,
)

texts = text_splitter.create_documents([page_text])
doc1  =  texts[1].page_content

vector_store.add_documents([texts[1]])

In [ ]:
import pprint

pprint.pp(docs[3].metadata)

In [ ]:
docs[3].metadata['new_field_1'] = 'value_1'
docs[3].metadata['new_field_2'] = 'value_2'
# Display the updated metadata
docs[3].metadata

In [ ]:
for index , row in filtered_df.head(1).iterrows():
  loader = PyPDFLoader(row.doc_link, mode="page")
  docs = loader.load()
  for idx , doc in enumerate(docs):
    doc.metadata['doc_name'] = row.doc_name
    doc.metadata['company'] = row.company
    doc.metadata['doc_period'] = row.doc_period
    doc.metadata['evidence_page_num'] = row.evidence[0]["evidence_page_num"]
    doc.metadata['page_number'] = idx
    docs[idx] = doc
    print(doc.metadata)